In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import joblib

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive/Colab\ Notebooks/Budget Tracking

/content/drive/MyDrive/Colab Notebooks/Budget Tracking


In [4]:
df = pd.read_csv("data.csv", encoding="utf-8")
print(df.head())

                                  input                label
0             Nhận lương 10tr ngày 24/9       income @ Lương
1            Nhận tiền nợ 5tr ngày 22/4  income @ Thu hồi nợ
2           Trả tiền spa 15tr ngày 16/1    expense @ Làm đẹp
3   Chi tiêu cho ăn trưa 20tr ngày 13/8    expense @ Ăn uống
4  Đặt hàng máy tính bảng 5tr ngày 3/12    expense @ Mua sắm


In [5]:
# Sử dụng TfidfVectorizer cho cột input
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["input"])

# Nhãn (label) là cột chứa chuỗi kết hợp (ví dụ: "income @ Lương")
y = df["label"]

# Tách dữ liệu thành tập train và test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
# Khởi tạo mô hình RandomForest ban đầu
rf = RandomForestClassifier(n_estimators=100, random_state=42)

# Tinh chỉnh siêu tham số với GridSearchCV
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}

grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)
print("Best Parameters:", grid_search.best_params_)
best_rf = grid_search.best_estimator_

# Kết hợp mô hình với VotingClassifier
model = VotingClassifier(
    estimators=[
        ('rf', best_rf),
        ('gb', GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42)),
        ('xgb', XGBClassifier(n_estimators=200, max_depth=5, random_state=42, use_label_encoder=False, eval_metric='logloss')),
        ('lr', LogisticRegression(max_iter=1000))
    ],
    voting='soft'
)
model.fit(X_train, y_train)

Best Parameters: {'max_depth': 20, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:15:22] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(max_depth=20,
                                                     max_features='log2',
                                                     random_state=42)),
                             ('gb',
                              GradientBoostingClassifier(max_depth=5,
                                                         n_estimators=200,
                                                         random_state=42)),
                             ('xgb',
                              XGBClassifier(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=None, device=None,
                                            early_stopping_ro...
                                            interaction_constraints=None,
                                            learning_rate=None, max_bin=None,
                                            max_cat_threshold=None,
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=5,
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=200, n_jobs=None,
                                            num_parallel_tree=None,
                                            random_state=42, ...)),
                             ('lr', LogisticRegression(max_iter=1000))],
                 voting='soft')

In [7]:
# Dự đoán trên tập test và đánh giá
y_pred = model.predict(X_test)
print(f"Độ chính xác: {accuracy_score(y_test, y_pred):.2f}")

Độ chính xác: 0.99


In [8]:
# Hàm dự đoán cho dữ liệu mới
def predict(raw_text):
    if not raw_text:
        return "Chưa xác định"
    processed_text = vectorizer.transform([raw_text])
    pred = model.predict(processed_text)[0]
    return pred

In [9]:
# Ví dụ sử dụng
test_text = "Nhận doanh thu quảng cáo"
predicted_label = predict(test_text)
print(f"Input: \"{test_text}\"")
print(f"Predicted Label: {predicted_label}")

Input: "Nhận doanh thu quảng cáo"
Predicted Label: income @ Kinh doanh


In [10]:
# Lưu model
joblib.dump(model, 'model.pkl')

# Lưu vectorizer nếu cần sử dụng lại cho quá trình dự đoán
joblib.dump(vectorizer, 'vectorizer.pkl')

['vectorizer.pkl']

In [25]:
def extract_info(text):
    # 📌 1. Extract date information first
    date_pattern = r"(?:ngày\s*)?(\d{1,2})[/-](\d{1,2})(?:[/-]\d{4})?|\b(\d{1,2})\s*tháng\s*(\d{1,2})\b"
    date_match = re.search(date_pattern, text, re.IGNORECASE)

    date_str = None
    if date_match:
        day = date_match.group(1) or date_match.group(3)
        month = date_match.group(2) or date_match.group(4)
        year = datetime.now().year
        date_str = f"{int(day):02d}/{int(month):02d}/{year}"
    else:
        month_only_pattern = r"tháng\s+(\d{1,2})"
        month_match = re.search(month_only_pattern, text, re.IGNORECASE)
        if month_match:
            month = int(month_match.group(1))
            day = datetime.now().day
            year = datetime.now().year
            date_str = f"{day:02d}/{month:02d}/{year}"
        else:
            date_str = datetime.now().strftime("%d/%m/%Y")

    # 📌 2. Extract money amount with improved pattern
    money_pattern = r"(\d+(?:[.,]\d{3})*(?:[.,]\d+)?|\d+)\s*(k|nghìn|ngàn|tr|triệu|m|vnđ|đồng)?(?:\s*/?\w*)?"
    money_with_phrase_pattern = r"(?:số tiền|giá|chi phí|tổng chi phí)\s*(\d+(?:[.,]\d{3})*(?:[.,]\d+)?|\d+)\s*(k|nghìn|ngàn|tr|triệu|m|vnđ|đồng)?(?:\s*/?\w*)?"
    number_word_pattern = r"(?:(\d+)\s*(triệu|nghìn|ngàn|m))"

    amount = 0

    # Try specific phrase pattern first
    money_with_phrase = re.search(money_with_phrase_pattern, text, re.IGNORECASE)
    if money_with_phrase:
        value, unit = money_with_phrase.groups()
        clean_value = value.replace(",", ".")
        if clean_value.count(".") > 1:
            clean_value = clean_value.replace(".", "", clean_value.count(".") - 1)
        try:
            numeric_value = float(clean_value)
            if unit:
                unit = unit.lower()
                if unit in ["k", "nghìn", "ngàn"]:
                    numeric_value *= 1_000
                elif unit in ["tr", "triệu", "m"]:
                    numeric_value *= 1_000_000
            amount = int(numeric_value)
        except ValueError:
            pass
    else:
        # Try number word pattern (e.g., "3 triệu", "5 ngàn", "2 m")
        number_word_match = re.search(number_word_pattern, text, re.IGNORECASE)
        if number_word_match:
            value, unit = number_word_match.groups()
            try:
                numeric_value = float(value)
                if unit.lower() in ["nghìn", "ngàn"]:
                    numeric_value *= 1_000
                elif unit.lower() in ["tr", "triệu", "m"]:
                    numeric_value *= 1_000_000
                amount = int(numeric_value)
            except ValueError:
                pass
        else:
            # Fallback to general money pattern
            money_matches = re.finditer(money_pattern, text, re.IGNORECASE)
            for match in money_matches:
                value, unit = match.groups()
                context_before = text[:match.start()].split()[-3:] if match.start() > 0 else []
                context_after = text[match.end():].split()[:3] if match.end() < len(text) else []
                context_str = " ".join(context_before + context_after).lower()

                # Skip if it’s a per-unit cost (e.g., "200k/tháng") but allow if it’s the main amount
                if "/tháng" in context_str or "/người" in context_str:
                    continue
                # Only skip numbers without units if they look like part of a date
                if not unit and re.search(r"^\d+[/-]\d+", text.strip()) and value in text.split()[0]:
                    continue
                if not unit and (f"tháng {value}" in text.lower() or f"tháng{value}" in text.lower()):
                    continue
                if not unit and "ngày" in context_before and re.match(r"\d+", value):
                    continue

                clean_value = value.replace(",", ".")
                if clean_value.count(".") > 1:
                    clean_value = clean_value.replace(".", "", clean_value.count(".") - 1)
                try:
                    numeric_value = float(clean_value)
                    if unit:
                        unit = unit.lower()
                        if unit in ["k", "nghìn", "ngàn"]:
                            numeric_value *= 1_000
                        elif unit in ["tr", "triệu", "m"]:
                            numeric_value *= 1_000_000
                    amount = int(numeric_value)
                    break  # Only take the first valid money amount
                except ValueError:
                    continue

    return date_str, amount

In [12]:
inputs = [
  "19/3 100000 ăn sáng ở Chane",
  "100k ăn sáng ở Chane vào ngày 19/3",
  "Hôm nay ăn sáng ở Chane hết 100k",
  "Đi Grab 50000 ngày 21/3",
  "Nhận lương 15m ngày 1/4",
  "200.000 vnđ tiền điện tháng 3",
  "3 triệu mua laptop ngày 15 tháng 4",
  "Mua sắm tại Vincom 500000 ngày 10/4",
  "Trả tiền nước tháng 4 là 150k",
  "Đi ăn tối ở nhà hàng 300k ngày 25/3",
  "Mua sách 100k ngày 5/4",
  "Trả tiền internet 200k ngày 15/3",
  "Nhận tiền lãi 1tr ngày 20/4",
  "Mua điện thoại 10tr ngày 28/3",
  "Đi du lịch 5tr ngày 1/5",
  "Trả tiền bảo hiểm 2tr ngày 10/4",
  "Mua thực phẩm 200k ngày 22/3",
  "Trả tiền học phí 5tr ngày 15/4",
  "Mua quần áo 800k ngày 12/4",
  "Đi xem phim 150k ngày 18/3",
  "Mua đồ dùng gia đình 300k ngày 8/4"
  "Mua mỹ phẩm 250k ngày 20/4",
  "Trả tiền thuê nhà 8tr ngày 1/5",
  "Đi ăn trưa ở quán 80k ngày 22/3",
  "Mua đồ chơi cho bé 120k ngày 15/4",
  "Trả tiền bảo trì xe 500k ngày 10/4",
  "Nhận tiền thưởng 2tr ngày 25/4",
  "Mua thực phẩm cho mèo 50k ngày 18/3",
  "Đi tập gym 200k/tháng ngày 1/4",
  "Mua đồ nội thất 1,5tr ngày 28/3",
  "Trả tiền học ngoại ngữ 3tr ngày 12/4"
  "Trả tiền học phí cho con học tiếng Anh tại trung tâm ngoại ngữ vào ngày 15/4 với số tiền 2,5 triệu đồng",
  "Mua một chiếc xe đạp điện giá 8 triệu đồng vào ngày 20/3",
  "Đi du lịch đến Đà Nẵng trong 3 ngày với tổng chi phí 10 triệu đồng, bắt đầu từ ngày 1/5",
  "Trả tiền bảo hiểm nhân thọ hàng năm vào ngày 10/4 với số tiền 5 triệu đồng",
  "Nhận tiền lãi từ việc đầu tư chứng khoán vào ngày 25/4 với số tiền 1,2 triệu đồng",
  "Mua một bộ sofa cho phòng khách giá 12 triệu đồng vào ngày 28/3",
  "Đi ăn tối tại nhà hàng cao cấp vào ngày 18/3 với chi phí 500k/người",
  "Trả tiền thuê nhà trọ hàng tháng vào ngày 1/4 với số tiền 3,5 triệu đồng",
  "Mua một chiếc máy giặt mới giá 7 triệu đồng vào ngày 22/3",
  "Nhận tiền thưởng cuối năm vào ngày 31/12 với số tiền 10 triệu đồng",
  "Trả tiền học phí đại học cho con vào ngày 15/9 với số tiền 20 triệu đồng",
  "Mua một chiếc máy tính bảng giá 5 triệu đồng vào ngày 10/4",
  "Đi tham gia một khóa học trực tuyến về lập trình vào ngày 1/5 với chi phí 2 triệu đồng",
  "Trả tiền bảo trì máy lạnh vào ngày 25/4 với số tiền 200k",
  "Mua một bộ đồ nội thất phòng ngủ giá 15 triệu đồng vào ngày 12/4",
  "Nhận tiền từ việc bán hàng trực tuyến vào ngày 20/4 với số tiền 1,5 triệu đồng",
  "Đi xem một trận đấu bóng đá vào ngày 15/4 với vé giá 1 triệu đồng",
  "Trả tiền phí đăng ký thi chứng chỉ tiếng Anh vào ngày 10/4 với số tiền 1,8 triệu đồng",
  "Mua một chiếc máy ảnh giá 10 triệu đồng vào ngày 28/3",
  "Nhận tiền từ việc cho thuê phòng trọ vào ngày 1/5 với số tiền 4 triệu đồng"
  "Trả tiền phí gia hạn bằng lái xe vào ngày 15/4 với số tiền 500k",
  "Mua một bộ dụng cụ làm vườn giá 800k vào ngày 22/3",
  "Đi tham gia một lớp học nấu ăn vào ngày 10/4 với chi phí 1,2 triệu đồng",
  "Nhận tiền từ việc bán đồ cũ vào ngày 25/4 với số tiền 300k",
  "Trả tiền phí đăng ký tham gia một sự kiện thể thao vào ngày 1/5 với số tiền 200k",
  "Mua một chiếc máy ép trái cây giá 1,5 triệu đồng vào ngày 28/3",
  "Đi du lịch đến Hà Nội trong 4 ngày với tổng chi phí 8 triệu đồng, bắt đầu từ ngày 15/4",
  "Trả tiền phí bảo trì máy tính vào ngày 20/4 với số tiền 150k",
  "Nhận tiền từ việc làm thêm vào ngày 30/4 với số tiền 2 triệu đồng",
  "Mua một bộ sách giáo khoa cho năm học mới giá 1 triệu đồng vào ngày 10/8",
  "Đi ăn sáng tại một quán cà phê vào ngày 18/3 với chi phí 100k/người",
  "Trả tiền phí tham gia một khóa học yoga vào ngày 1/4 với số tiền 1,8 triệu đồng",
  "Mua một chiếc đồng hồ giá 3 triệu đồng vào ngày 22/3",
  "Nhận tiền từ việc hoàn thành một dự án freelance vào ngày 25/4 với số tiền 5 triệu đồng",
  "Trả tiền phí đăng ký thi một chứng chỉ chuyên môn vào ngày 10/4 với số tiền 2,5 triệu đồng",
  "Mua một bộ thiết bị âm thanh giá 6 triệu đồng vào ngày 28/3",
  "Đi tham gia một hội thảo về công nghệ vào ngày 15/4 với vé giá 1,5 triệu đồng",
  "Trả tiền phí bảo trì xe máy vào ngày 20/4 với số tiền 100k",
  "Nhận tiền từ việc bán một sản phẩm trực tuyến vào ngày 1/5 với số tiền 1 triệu đồng"
  "Trả tiền phí gia hạn thẻ tín dụng vào ngày 15/4 với số tiền 200k",
  "Mua một chiếc máy hút bụi giá 2 triệu đồng vào ngày 22/3",
  "Đi tham gia một lớp học vẽ vào ngày 10/4 với chi phí 1,5 triệu đồng",
  "Nhận tiền từ việc bán một chiếc xe đạp cũ vào ngày 25/4 với số tiền 1,2 triệu đồng",
  "Trả tiền phí đăng ký tham gia một cuộc thi nấu ăn vào ngày 1/5 với số tiền 300k",
  "Mua một bộ dụng cụ làm đẹp giá 1 triệu đồng vào ngày 28/3",
  "Đi du lịch đến Nha Trang trong 5 ngày với tổng chi phí 12 triệu đồng, bắt đầu từ ngày 15/4",
  "Trả tiền phí bảo trì điều hòa vào ngày 20/4 với số tiền 250k",
  "Nhận tiền từ việc làm việc bán thời gian vào ngày 30/4 với số tiền 3 triệu đồng",
  "Mua một bộ sách tham khảo cho kỳ thi vào ngày 10/8 với giá 800k",
  "Đi ăn trưa tại một nhà hàng vào ngày 18/3 với chi phí 150k/người",
  "Trả tiền phí tham gia một khóa học thiết kế đồ họa vào ngày 1/4 với số tiền 4 triệu đồng",
  "Mua một chiếc loa di động giá 1,8 triệu đồng vào ngày 22/3",
  "Nhận tiền từ việc hoàn thành một dự án thiết kế vào ngày 25/4 với số tiền 6 triệu đồng",
  "Trả tiền phí đăng ký thi một chứng chỉ ngoại ngữ vào ngày 10/4 với số tiền 3,5 triệu đồng",
  "Mua một bộ thiết bị thể thao giá 4 triệu đồng vào ngày 28/3",
  "Đi tham gia một hội thảo về kinh doanh vào ngày 15/4 với vé giá 2,5 triệu đồng",
  "Trả tiền phí bảo trì máy in vào ngày 20/4 với số tiền 150k",
  "Nhận tiền từ việc bán một sản phẩm handmade vào ngày 1/5 với số tiền 500k"
  "Trả tiền phí gia hạn bảo hiểm sức khỏe vào ngày 15/4 với số tiền 1 triệu đồng",
  "Mua một chiếc máy xay sinh tố giá 800k vào ngày 22/3",
  "Đi tham gia một lớp học nấu ăn chay vào ngày 10/4 với chi phí 1,2 triệu đồng",
  "Nhận tiền từ việc bán một chiếc điện thoại cũ vào ngày 25/4 với số tiền 2 triệu đồng",
  "Trả tiền phí đăng ký tham gia một cuộc thi thời trang vào ngày 1/5 với số tiền 500k",
  "Mua một bộ dụng cụ làm tóc giá 1,5 triệu đồng vào ngày 28/3",
  "Đi du lịch đến Đà Lạt trong 4 ngày với tổng chi phí 10 triệu đồng, bắt đầu từ ngày 15/4",
  "Trả tiền phí bảo trì máy giặt vào ngày 20/4 với số tiền 300k",
  "Nhận tiền từ việc làm việc part-time vào ngày 30/4 với số tiền 2,5 triệu đồng",
  "Mua một bộ sách tham khảo cho kỳ thi đại học vào ngày 10/8 với giá 1,2 triệu đồng",
  "Đi ăn tối tại một nhà hàng cao cấp vào ngày 18/3 với chi phí 1 triệu đồng/người",
  "Trả tiền phí tham gia một khóa học lập trình vào ngày 1/4 với số tiền 5 triệu đồng",
  "Mua một chiếc đồng hồ thông minh giá 3,5 triệu đồng vào ngày 22/3",
  "Nhận tiền từ việc hoàn thành một dự án thiết kế web vào ngày 25/4 với số tiền 8 triệu đồng",
  "Trả tiền phí đăng ký thi một chứng chỉ chuyên môn vào ngày 10/4 với số tiền 4 triệu đồng",
  "Mua một bộ thiết bị âm thanh giá 7 triệu đồng vào ngày 28/3",
  "Đi tham gia một hội thảo về công nghệ thông tin vào ngày 15/4 với vé giá 3 triệu đồng",
  "Trả tiền phí bảo trì máy tính vào ngày 20/4 với số tiền 400k",
  "Nhận tiền từ việc bán một sản phẩm trực tuyến vào ngày 1/5 với số tiền 1,5 triệu đồng",
  "Trả tiền phí gia hạn bảo hiểm nhân thọ vào ngày 15/4 với số tiền 2 triệu đồng",
  "Mua một chiếc máy ép trái cây giá 1,2 triệu đồng vào ngày 22/3",
  "Đi tham gia một lớp học nấu ăn vào ngày 10/4 với chi phí 2 triệu đồng",
  "Nhận tiền từ việc bán một chiếc ô tô cũ vào ngày 25/4 với số tiền 200 triệu đồng",
  "Trả tiền phí đăng ký tham gia một cuộc thi âm nhạc vào ngày 1/5 với số tiền 1 triệu đồng",
  "Mua một bộ dụng cụ làm tóc giá 3 triệu đồng vào ngày 28/3",
  "Đi du lịch đến Bali trong 6 ngày với tổng chi phí 30 triệu đồng, bắt đầu từ ngày 15/4",
  "Trả tiền phí bảo trì máy giặt vào ngày 20/4 với số tiền 400k",
  "Nhận tiền từ việc làm việc part-time vào ngày 30/4 với số tiền 4 triệu đồng",
  "Mua một bộ sách tham khảo cho kỳ thi đại học vào ngày 10/8 với giá 2 triệu đồng",
  "Đi ăn tối tại một nhà hàng cao cấp vào ngày 18/3 với chi phí 1,5 triệu đồng/người",
  "Trả tiền phí tham gia một khóa học lập trình vào ngày 1/4 với số tiền 7 triệu đồng",
  "Mua một chiếc đồng hồ thông minh giá 4 triệu đồng vào ngày 22/3",
  "Nhận tiền từ việc hoàn thành một dự án thiết kế vào ngày 25/4 với số tiền 10 triệu đồng",
  "Trả tiền phí đăng ký thi một chứng chỉ chuyên môn vào ngày 10/4 với số tiền 6 triệu đồng",
  "Mua một bộ thiết bị âm thanh giá 9 triệu đồng vào ngày 28/3",
  "Đi tham gia một hội thảo về công nghệ thông tin vào ngày 15/4 với vé giá 5 triệu đồng",
  "Trả tiền phí bảo trì máy tính vào ngày 20/4 với số tiền 500k",
  "Nhận tiền từ việc bán một sản phẩm trực tuyến vào ngày 1/5 với số tiền 2 triệu đồng",

  "Trả tiền phí gia hạn thẻ tín dụng vào ngày 15/4 với số tiền 500k",
  "Mua một chiếc máy sấy tóc giá 800k vào ngày 22/3",
  "Đi tham gia một lớp học yoga vào ngày 10/4 với chi phí 2,5 triệu đồng",
  "Nhận tiền từ việc bán một chiếc xe đạp điện cũ vào ngày 25/4 với số tiền 3 triệu đồng",
  "Trả tiền phí đăng ký tham gia một cuộc thi thể thao vào ngày 1/5 với số tiền 1,2 triệu đồng",
  "Mua một bộ dụng cụ làm đẹp giá 3,5 triệu đồng vào ngày 28/3",
  "Đi du lịch đến Singapore trong 4 ngày với tổng chi phí 15 triệu đồng, bắt đầu từ ngày 15/4",
  "Trả tiền phí bảo trì điều hòa vào ngày 20/4 với số tiền 450k",
  "Nhận tiền từ việc làm việc full-time vào ngày 30/4 với số tiền 12 triệu đồng",
  "Mua một bộ sách tham khảo cho kỳ thi vào ngày 10/8 với giá 2,5 triệu đồng",
  "Đi ăn sáng tại một quán cà phê vào ngày 18/3 với chi phí 100k/người",
  "Trả tiền phí tham gia một khóa học thiết kế vào ngày 1/4 với số tiền 8 triệu đồng",
  "Mua một chiếc loa Bluetooth giá 3,5 triệu đồng vào ngày 22/3",
  "Nhận tiền từ việc hoàn thành một dự án xây dựng vào ngày 25/4 với số tiền 60 triệu đồng",
  "Trả tiền phí đăng ký thi một chứng chỉ ngoại ngữ vào ngày 10/4 với số tiền 7 triệu đồng",
  "Mua một bộ thiết bị thể thao giá 10 triệu đồng vào ngày 28/3",
  "Đi tham gia một hội thảo về kinh doanh vào ngày 15/4 với vé giá 6 triệu đồng",
  "Trả tiền phí bảo trì máy in vào ngày 20/4 với số tiền 250k",
  "Nhận tiền từ việc bán một sản phẩm handmade vào ngày 1/5 với số tiền 1,5 triệu đồng",
  "Trả tiền phí gia hạn thẻ thành viên vào ngày 15/4 với số tiền 400k",
  "Mua một chiếc máy sấy tóc giá 1 triệu đồng vào ngày 22/3",
  "Đi tham gia một lớp học yoga vào ngày 10/4 với chi phí 3,5 triệu đồng",
  "Nhận tiền từ việc bán một chiếc xe đạp điện cũ vào ngày 25/4 với số tiền 4 triệu đồng",
  "Trả tiền phí đăng ký tham gia một cuộc thi thể thao vào ngày 1/5 với số tiền 1,8 triệu đồng",
  "Mua một bộ dụng cụ làm đẹp giá 5 triệu đồng vào ngày 28/3",
  "Đi du lịch đến Malaysia trong 4 ngày với tổng chi phí 12 triệu đồng, bắt đầu từ ngày 15/4",
  "Trả tiền phí bảo trì điều hòa vào ngày 20/4 với số tiền 550k",
  "Nhận tiền từ việc làm việc full-time vào ngày 30/4 với số tiền 15 triệu đồng",
  "Mua một bộ sách tham khảo cho kỳ thi vào ngày 10/8 với giá 3,5 triệu đồng",
  "Đi ăn sáng tại một quán cà phê vào ngày 18/3 với chi phí 120k/người",
  "Trả tiền phí tham gia một khóa học thiết kế vào ngày 1/4 với số tiền 10 triệu đồng",
  "Mua một chiếc loa Bluetooth giá 4,5 triệu đồng vào ngày 22/3",
  "Nhận tiền từ việc hoàn thành một dự án xây dựng vào ngày 25/4 với số tiền 80 triệu đồng",
  "Trả tiền phí đăng ký thi một chứng chỉ ngoại ngữ vào ngày 10/4 với số tiền 9 triệu đồng",
  "Mua một bộ thiết bị thể thao giá 12 triệu đồng vào ngày 28/3",
  "Đi tham gia một hội thảo về kinh doanh vào ngày 15/4 với vé giá 8 triệu đồng",
  "Trả tiền phí bảo trì máy in vào ngày 20/4 với số tiền 300k",
  "Nhận tiền từ việc bán một sản phẩm handmade vào ngày 1/5 với số tiền 2 triệu đồng",
  "Trả tiền phí gia hạn bảo hiểm nhân thọ vào ngày 15/4 với số tiền 3 triệu đồng",
  "Mua một chiếc máy ép trái cây giá 2,5 triệu đồng vào ngày 22/3",
  "Đi tham gia một lớp học nấu ăn vào ngày 10/4 với chi phí 4 triệu đồng",
  "Nhận tiền từ việc bán một chiếc ô tô cũ vào ngày 25/4 với số tiền 400 triệu đồng",
  "Trả tiền phí đăng ký tham gia một cuộc thi âm nhạc vào ngày 1/5 với số tiền 2,5 triệu đồng",
  "Mua một bộ dụng cụ làm tóc giá 6 triệu đồng vào ngày 28/3",
  "Đi du lịch đến Campuchia trong 5 ngày với tổng chi phí 20 triệu đồng, bắt đầu từ ngày 15/4",
  "Trả tiền phí bảo trì máy giặt vào ngày 20/4 với số tiền 600k",
  "Nhận tiền từ việc làm việc part-time vào ngày 30/4 với số tiền 6 triệu đồng",
  "Mua một bộ sách tham khảo cho kỳ thi đại học vào ngày 10/8 với giá 4 triệu đồng",
  "Đi ăn tối tại một nhà hàng cao cấp vào ngày 18/3 với chi phí 3 triệu đồng/người",
  "Trả tiền phí tham gia một khóa học lập trình vào ngày 1/4 với số tiền 11 triệu đồng",
  "Mua một chiếc đồng hồ thông minh giá 6 triệu đồng vào ngày 22/3",
  "Nhận tiền từ việc hoàn thành một dự án thiết kế vào ngày 25/4 với số tiền 15 triệu đồng",
  "Trả tiền phí đăng ký thi một chứng chỉ chuyên môn vào ngày 10/4 với số tiền 10 triệu đồng",
  "Mua một bộ thiết bị âm thanh giá 13 triệu đồng vào ngày 28/3",
  "Đi tham gia một hội thảo về công nghệ thông tin vào ngày 15/4 với vé giá 9 triệu đồng",
  "Trả tiền phí bảo trì máy tính vào ngày 20/4 với số tiền 700k",
  "Nhận tiền từ việc bán một sản phẩm trực tuyến vào ngày 1/5 với số tiền 4 triệu đồng"
]

In [13]:
# vectorizer = joblib.load('vectorizer.pkl')
# model = joblib.load('model.pkl')

In [27]:
for input in inputs:
  info = extract_info(input)
  print(info)
  print(input)
  print(predict(input))

('19/03/2025', 100000)
19/3 100k ăn sáng ở Chane
expense @ Ăn uống
('19/03/2025', 100000)
100k ăn sáng ở Chane vào ngày 19/3
expense @ Ăn uống
('31/03/2025', 100000)
Hôm nay ăn sáng ở Chane hết 100k
expense @ Ăn uống
('21/03/2025', 50000)
Đi Grab 50k ngày 21/3
expense @ Di chuyển
('01/04/2025', 15000000)
Nhận lương 15tr ngày 1/4
income @ Lương
('31/03/2025', 200)
200.000 vnđ tiền điện tháng 3
expense @ Di chuyển
('15/04/2025', 3000000)
3 triệu mua laptop ngày 15 tháng 4
expense @ Mua sắm
('10/04/2025', 500000)
Mua sắm tại Vincom 500k ngày 10/4
expense @ Di chuyển
('31/04/2025', 150000)
Trả tiền nước tháng 4 là 150k
expense @ Hóa đơn
('25/03/2025', 300000)
Đi ăn tối ở nhà hàng 300k ngày 25/3
expense @ Ăn uống
('05/04/2025', 100000)
Mua sách 100k ngày 5/4
expense @ Mua sắm
('15/03/2025', 200000)
Trả tiền internet 200k ngày 15/3
expense @ Hóa đơn
('20/04/2025', 1000000)
Nhận tiền lãi 1tr ngày 20/4
income @ Thu hồi nợ
('28/03/2025', 10000000)
Mua điện thoại 10tr ngày 28/3
expense @ Mua sắm